In [ ]:
!pip install -q -U torchao

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

print("Base model loaded:", MODEL_NAME)

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel
import copy

LORA_RANK = 8
LORA_ALPHA = 16

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

toy_tasks = {
    "grammar_en": [
        ("Fix the grammar: She go to school every day.", "She goes to school every day."),
        ("Fix the grammar: He don't like apples.", "He doesn't like apples."),
        ("Fix the grammar: They was late.", "They were late."),
    ],
    "summarize_en": [
        ("Summarize: The cat sat on the mat and slept for hours in the warm sun.",
         "A cat slept on a mat in the sun."),
        ("Summarize: The team worked all night to finish the project before the deadline.",
         "The team worked overnight to meet the deadline."),
    ],
    "translate_fr": [
        ("Translate to French: Good morning, how are you?", "Bonjour, comment ça va ?"),
        ("Translate to French: I would like a coffee.", "Je voudrais un café."),
    ],
    "grammar_de": [
        ("Fix the grammar: Er gehen zur Schule.", "Er geht zur Schule."),
        ("Fix the grammar: Wir hat keine Zeit.", "Wir haben keine Zeit."),
    ],
}

print("Toy tasks defined:", list(toy_tasks.keys()))

In [ ]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=64):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prompt, target = self.pairs[idx]
        text = f"{prompt}\nAnswer: {target}{self.tokenizer.eos_token}"
        enc = self.tokenizer(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

def train_toy_lora(task_name, pairs, epochs=15, lr=1e-3):
    model = get_peft_model(copy.deepcopy(base_model), lora_config)
    model.to(device)
    model.train()

    ds = ToyDataset(pairs, tokenizer)
    dl = DataLoader(ds, batch_size=2, shuffle=True)
    optim = AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        for batch in dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            optim.step()
            optim.zero_grad()
    print(f"[{task_name}] final loss: {loss.item():.4f}")

    lora_state = {k: v.detach().clone().cpu()
                  for k, v in model.state_dict().items() if "lora_" in k}
    return lora_state

trained_loras = {}
for name, pairs in toy_tasks.items():
    trained_loras[name] = train_toy_lora(name, pairs)

print("Trained LoRAs:", list(trained_loras.keys()))

In [ ]:
import torch.nn.functional as F

def flatten_lora_deltas(lora_state):
    modules = {}
    for k, v in lora_state.items():
        prefix = k.rsplit(".lora_", 1)[0]
        kind = "A" if "lora_A" in k else "B"
        modules.setdefault(prefix, {})[kind] = v

    flat_vectors = []
    for prefix, d in sorted(modules.items()):
        if "A" in d and "B" in d:
            delta = d["B"].float() @ d["A"].float()
            flat_vectors.append(delta.flatten())
    return torch.cat(flat_vectors)

def lora_similarity(lora_state_1, lora_state_2):
    v1 = flatten_lora_deltas(lora_state_1)
    v2 = flatten_lora_deltas(lora_state_2)
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

def merge_loras(stored_lora, incoming_lora, n_merged_so_far):
    merged = {}
    for k in stored_lora.keys():
        merged[k] = (incoming_lora[k] + n_merged_so_far * stored_lora[k]) / (n_merged_so_far + 1)
    return merged

print("K-Merge core functions ready.")

In [ ]:
K = 3

collection = {}
history = {}
task_to_slot = {}

task_order = list(trained_loras.keys())
print("Arrival order:", task_order)

for t, task_name in enumerate(task_order):
    incoming = trained_loras[task_name]
    print(f"\n--- t={t}: '{task_name}' arrives ---")

    if len(collection) < K:
        slot_name = task_name
        collection[slot_name] = incoming
        history[slot_name] = {task_name}
        task_to_slot[task_name] = slot_name
        print(f"  Stored in new slot '{slot_name}' ({len(collection)}/{K} used)")
    else:
        sims = {slot: lora_similarity(collection[slot], incoming) for slot in collection}
        best_slot = max(sims, key=sims.get)
        print(f"  Storage full. Similarities: { {k: round(v,3) for k,v in sims.items()} }")
        print(f"  Merging '{task_name}' into slot '{best_slot}' (sim={sims[best_slot]:.3f})")

        n_prev = len(history[best_slot])
        collection[best_slot] = merge_loras(collection[best_slot], incoming, n_prev)
        history[best_slot].add(task_name)
        task_to_slot[task_name] = best_slot

print("\n=== Final state ===")
for slot, tasks in history.items():
    print(f"Slot '{slot}' serves tasks: {tasks}")

In [ ]:
def load_lora_state_into_model(lora_state):
    model = get_peft_model(copy.deepcopy(base_model), lora_config)
    model.to(device)
    sd = model.state_dict()
    for k, v in lora_state.items():
        sd[k] = v.to(device).to(sd[k].dtype)
    model.load_state_dict(sd)
    model.eval()
    return model

def generate(model, prompt, max_new_tokens=30):
    inputs = tokenizer(prompt + "\nAnswer:", return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

test_prompts = {
    "grammar_en": "Fix the grammar: She don't like the movie.",
    "summarize_en": "Summarize: The dog ran across the park chasing a red ball for a long time.",
    "translate_fr": "Translate to French: Where is the train station?",
    "grammar_de": "Fix the grammar: Sie gehen nicht zur Arbeit heute.",
}

for task_name, prompt in test_prompts.items():
    slot = task_to_slot[task_name]
    model = load_lora_state_into_model(collection[slot])
    output = generate(model, prompt)
    print(f"[{task_name}] (served by slot '{slot}')")
    print(f"  Prompt: {prompt}")
    print(f"  Output: {output}\n")

In [ ]:
import random

en_eval = [
    ("Fix the grammar: He don't know the answer.", "He doesn't know the answer."),
    ("Fix the grammar: She have finished her homework.", "She has finished her homework."),
    ("Fix the grammar: The dogs is barking loudly.", "The dogs are barking loudly."),
    ("Fix the grammar: I has a car.", "I have a car."),
    ("Fix the grammar: We was happy.", "We were happy."),
    ("Fix the grammar: He like pizza.", "He likes pizza."),
]

de_eval = [
    ("Fix the grammar: Ich habe nicht Zeit.", "Ich habe keine Zeit."),
    ("Fix the grammar: Du bist mein beste Freund.", "Du bist mein bester Freund."),
    ("Fix the grammar: Wir gehst ins Kino.", "Wir gehen ins Kino."),
    ("Fix the grammar: Er haben ein Auto.", "Er hat ein Auto."),
    ("Fix the grammar: Sie ist mein Schwester.", "Sie ist meine Schwester."),
    ("Fix the grammar: Ich kann nicht schwimmen gut.", "Ich kann nicht gut schwimmen."),
]

def clean(s):
    return "".join(c.lower() for c in s if c.isalnum() or c.isspace()).split()

def score(pred, target):
    p = clean(pred)
    t = clean(target)
    if not p or not t:
        return 0
    common = set(p) & set(t)
    if not common:
        return 0
    overlap = sum(min(p.count(w), t.count(w)) for w in common)
    prec = overlap / len(p)
    rec = overlap / len(t)
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0

def run_eval(model, eval_set):
    total = 0
    for prompt, target in eval_set:
        out = generate(model, prompt)
        total += score(out, target)
    return total / len(eval_set)

print("eval sets ready:", len(en_eval), "english,", len(de_eval), "german")

In [ ]:
import torch
import gc

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

def run_one_trial(seed, order):
    random.seed(seed)
    torch.manual_seed(seed)

    en_lora = train_toy_lora("grammar_en_p1", toy_tasks["grammar_en"])
    de_lora = train_toy_lora("grammar_de_p1", toy_tasks["grammar_de"])
    cleanup()

    en_model = load_lora_state_into_model(en_lora)
    en_before = run_eval(en_model, en_eval)
    del en_model
    cleanup()

    de_model = load_lora_state_into_model(de_lora)
    de_before = run_eval(de_model, de_eval)
    del de_model
    cleanup()

    if order == "en_first":
        merged = merge_loras(en_lora, de_lora, 1)
    else:
        merged = merge_loras(de_lora, en_lora, 1)

    merged_model = load_lora_state_into_model(merged)
    en_after = run_eval(merged_model, en_eval)
    de_after = run_eval(merged_model, de_eval)
    del merged_model
    cleanup()

    del en_lora, de_lora, merged
    cleanup()

    return {
        "seed": seed,
        "order": order,
        "en_before": round(en_before, 3),
        "en_after": round(en_after, 3),
        "de_before": round(de_before, 3),
        "de_after": round(de_after, 3),
    }

print("trial function ready (memory-safe version)")

In [ ]:
import difflib

# New: French grammar correction toy task (training data)
toy_tasks["grammar_fr"] = [
    ("Fix the grammar: Il ne sais pas.", "Il ne sait pas."),
    ("Fix the grammar: Nous avons manger.", "Nous avons mangé."),
    ("Fix the grammar: Elle est aller au marché.", "Elle est allée au marché."),
]

# Bigger held-out eval sets (separate from training)
en_eval = [
    ("Fix the grammar: He don't know the answer.", "He doesn't know the answer."),
    ("Fix the grammar: She have finished her homework.", "She has finished her homework."),
    ("Fix the grammar: The dogs is barking loudly.", "The dogs are barking loudly."),
    ("Fix the grammar: I has a car.", "I have a car."),
    ("Fix the grammar: We was happy.", "We were happy."),
    ("Fix the grammar: He like pizza.", "He likes pizza."),
    ("Fix the grammar: They is playing outside.", "They are playing outside."),
    ("Fix the grammar: It don't work.", "It doesn't work."),
    ("Fix the grammar: She go to work by bus.", "She goes to work by bus."),
    ("Fix the grammar: We has two cats.", "We have two cats."),
]

de_eval = [
    ("Fix the grammar: Ich habe nicht Zeit.", "Ich habe keine Zeit."),
    ("Fix the grammar: Du bist mein beste Freund.", "Du bist mein bester Freund."),
    ("Fix the grammar: Wir gehst ins Kino.", "Wir gehen ins Kino."),
    ("Fix the grammar: Er haben ein Auto.", "Er hat ein Auto."),
    ("Fix the grammar: Sie ist mein Schwester.", "Sie ist meine Schwester."),
    ("Fix the grammar: Ich kann nicht schwimmen gut.", "Ich kann nicht gut schwimmen."),
    ("Fix the grammar: Das ist meine Bruder.", "Das ist mein Bruder."),
    ("Fix the grammar: Wir haben kein Auto.", "Wir haben kein Auto."),
    ("Fix the grammar: Sie kommt aus Deutschland nicht.", "Sie kommt nicht aus Deutschland."),
    ("Fix the grammar: Ich mag nicht Kaffee.", "Ich mag keinen Kaffee."),
]

fr_eval = [
    ("Fix the grammar: Il ne sais pas la réponse.", "Il ne sait pas la réponse."),
    ("Fix the grammar: Elle a manger une pomme.", "Elle a mangé une pomme."),
    ("Fix the grammar: Nous sommes aller au parc.", "Nous sommes allés au parc."),
    ("Fix the grammar: Je suis content de te voire.", "Je suis content de te voir."),
    ("Fix the grammar: Ils a fini le travail.", "Ils ont fini le travail."),
    ("Fix the grammar: Elle est parti hier.", "Elle est partie hier."),
    ("Fix the grammar: Tu as bien dormis?", "Tu as bien dormi?"),
    ("Fix the grammar: Nous avons vu le chien noire.", "Nous avons vu le chien noir."),
    ("Fix the grammar: Il fait ses devoir.", "Il fait ses devoirs."),
    ("Fix the grammar: Elle chante très bein.", "Elle chante très bien."),
]

def exact_match(pred, target):
    return 1.0 if clean(pred) == clean(target) else 0.0

def edit_accuracy(pred, target):
    # normalized similarity based on character-level edit distance
    ratio = difflib.SequenceMatcher(None, pred.strip().lower(), target.strip().lower()).ratio()
    return ratio

def run_eval_detailed(model, eval_set, tag=""):
    rows = []
    for prompt, target in eval_set:
        out = generate(model, prompt)
        rows.append({
            "task": tag,
            "prompt": prompt,
            "expected": target,
            "prediction": out,
            "f1": round(score(out, target), 3),
            "exact_match": exact_match(out, target),
            "edit_acc": round(edit_accuracy(out, target), 3),
        })
    avg_f1 = sum(r["f1"] for r in rows) / len(rows)
    avg_em = sum(r["exact_match"] for r in rows) / len(rows)
    avg_edit = sum(r["edit_acc"] for r in rows) / len(rows)
    return rows, {"avg_f1": round(avg_f1,3), "avg_exact_match": round(avg_em,3), "avg_edit_acc": round(avg_edit,3)}

print("Phase 1.5 setup ready: 3 languages, bigger eval sets, better metrics")

In [ ]:
import torch
import gc

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

def run_arrival_sequence(seed, order_names, K=2):
    """
    order_names: list like ["grammar_en", "grammar_de", "grammar_fr"]
    Trains all adapters fresh, then feeds them in the given order into a
    K-slot storage, merging when full. Evaluates ALL previously-seen
    languages after every single arrival step.
    """
    random.seed(seed)
    torch.manual_seed(seed)

    eval_map = {"grammar_en": en_eval, "grammar_de": de_eval, "grammar_fr": fr_eval}

    # train fresh adapters for this seed
    loras = {}
    for name in order_names:
        loras[name] = train_toy_lora(name, toy_tasks[name])
    cleanup()

    collection = {}
    history = {}
    step_log = []

    for t, name in enumerate(order_names):
        incoming = loras[name]

        if len(collection) < K:
            slot = name
            collection[slot] = incoming
            history[slot] = {name}
        else:
            sims = {s: lora_similarity(collection[s], incoming) for s in collection}
            best_slot = max(sims, key=sims.get)
            n_prev = len(history[best_slot])
            collection[best_slot] = merge_loras(collection[best_slot], incoming, n_prev)
            history[best_slot].add(name)

        # evaluate every task seen SO FAR, using whichever slot currently serves it
        task_to_slot_now = {}
        for slot, tasks in history.items():
            for task in tasks:
                task_to_slot_now[task] = slot

        seen_so_far = order_names[:t+1]
        step_scores = {}
        for task in seen_so_far:
            slot = task_to_slot_now[task]
            model = load_lora_state_into_model(collection[slot])
            _, summary = run_eval_detailed(model, eval_map[task], tag=task)
            step_scores[task] = summary["avg_f1"]
            del model
            cleanup()

        step_log.append({
            "seed": seed,
            "step": t,
            "arrived": name,
            "scores_so_far": step_scores,
            "avg": round(sum(step_scores.values())/len(step_scores), 3),
            "worst": round(min(step_scores.values()), 3),
        })

    del loras, collection
    cleanup()
    return step_log

print("arrival-order experiment function ready")

In [ ]:
orders_to_test = [
    ["grammar_en", "grammar_de", "grammar_fr"],
    ["grammar_fr", "grammar_de", "grammar_en"],
    ["grammar_de", "grammar_fr", "grammar_en"],
]

all_logs = []
for seed in [0, 1]:
    for order in orders_to_test:
        log = run_arrival_sequence(seed, order, K=2)
        all_logs.extend(log)
        print(f"\nseed={seed}, order={order}")
        for row in log:
            print(" ", row)

In [ ]:
random.seed(0)
torch.manual_seed(0)

lora_en = train_toy_lora("grammar_en", toy_tasks["grammar_en"])
lora_de = train_toy_lora("grammar_de", toy_tasks["grammar_de"])
lora_fr = train_toy_lora("grammar_fr", toy_tasks["grammar_fr"])

sim_en_de = lora_similarity(lora_en, lora_de)
sim_en_fr = lora_similarity(lora_en, lora_fr)
sim_de_fr = lora_similarity(lora_de, lora_fr)

print("sim(en, de):", round(sim_en_de, 4))
print("sim(en, fr):", round(sim_en_fr, 4))
print("sim(de, fr):", round(sim_de_fr, 4))

In [ ]:
import pandas as pd

def run_arrival_sequence_logged(seed, order_names, K=2):
    """Same as before, but also returns full prediction rows for CSV export."""
    random.seed(seed)
    torch.manual_seed(seed)

    eval_map = {"grammar_en": en_eval, "grammar_de": de_eval, "grammar_fr": fr_eval}

    loras = {}
    for name in order_names:
        loras[name] = train_toy_lora(name, toy_tasks[name])
    cleanup()

    collection = {}
    history = {}
    step_log = []
    all_prediction_rows = []

    for t, name in enumerate(order_names):
        incoming = loras[name]
        if len(collection) < K:
            slot = name
            collection[slot] = incoming
            history[slot] = {name}
        else:
            sims = {s: lora_similarity(collection[s], incoming) for s in collection}
            best_slot = max(sims, key=sims.get)
            n_prev = len(history[best_slot])
            collection[best_slot] = merge_loras(collection[best_slot], incoming, n_prev)
            history[best_slot].add(name)

        task_to_slot_now = {}
        for slot, tasks in history.items():
            for task in tasks:
                task_to_slot_now[task] = slot

        seen_so_far = order_names[:t+1]
        step_scores = {}
        for task in seen_so_far:
            slot = task_to_slot_now[task]
            model = load_lora_state_into_model(collection[slot])
            rows, summary = run_eval_detailed(model, eval_map[task], tag=f"seed{seed}_{'-'.join(order_names)}_step{t}_{task}")
            for r in rows:
                r["seed"] = seed
                r["order"] = "-".join(order_names)
                r["step"] = t
            all_prediction_rows.extend(rows)
            step_scores[task] = summary["avg_f1"]
            del model
            cleanup()

        step_log.append({
            "seed": seed, "step": t, "arrived": name,
            "scores_so_far": step_scores,
            "avg": round(sum(step_scores.values())/len(step_scores), 3),
            "worst": round(min(step_scores.values()), 3),
        })

    del loras, collection
    cleanup()
    return step_log, all_prediction_rows

# Redo the unstable run (seed 1, en->de->fr had a bad French training loss earlier)
fixed_log, fixed_predictions = run_arrival_sequence_logged(1, ["grammar_en", "grammar_de", "grammar_fr"], K=2)
for row in fixed_log:
    print(row)

pd.DataFrame(fixed_predictions).to_csv("predictions_seed1_en_de_fr.csv", index=False)
print("saved predictions CSV")

In [ ]:
import pandas as pd

def run_arrival_sequence_logged(seed, order_names, K=2):
    """Same as before, but also returns full prediction rows for CSV export."""
    random.seed(seed)
    torch.manual_seed(seed)

    eval_map = {"grammar_en": en_eval, "grammar_de": de_eval, "grammar_fr": fr_eval}

    loras = {}
    for name in order_names:
        loras[name] = train_toy_lora(name, toy_tasks[name])
    cleanup()

    collection = {}
    history = {}
    step_log = []
    all_prediction_rows = []

    for t, name in enumerate(order_names):
        incoming = loras[name]
        if len(collection) < K:
            slot = name
            collection[slot] = incoming
            history[slot] = {name}
        else:
            sims = {s: lora_similarity(collection[s], incoming) for s in collection}
            best_slot = max(sims, key=sims.get)
            n_prev = len(history[best_slot])
            collection[best_slot] = merge_loras(collection[best_slot], incoming, n_prev)
            history[best_slot].add(name)

        task_to_slot_now = {}
        for slot, tasks in history.items():
            for task in tasks:
                task_to_slot_now[task] = slot

        seen_so_far = order_names[:t+1]
        step_scores = {}
        for task in seen_so_far:
            slot = task_to_slot_now[task]
            model = load_lora_state_into_model(collection[slot])
            rows, summary = run_eval_detailed(model, eval_map[task], tag=f"seed{seed}_{'-'.join(order_names)}_step{t}_{task}")
            for r in rows:
                r["seed"] = seed
                r["order"] = "-".join(order_names)
                r["step"] = t
            all_prediction_rows.extend(rows)
            step_scores[task] = summary["avg_f1"]
            del model
            cleanup()

        step_log.append({
            "seed": seed, "step": t, "arrived": name,
            "scores_so_far": step_scores,
            "avg": round(sum(step_scores.values())/len(step_scores), 3),
            "worst": round(min(step_scores.values()), 3),
        })

    del loras, collection
    cleanup()
    return step_log, all_prediction_rows

# Redo the unstable run (seed 1, en->de->fr had a bad French training loss earlier)
fixed_log, fixed_predictions = run_arrival_sequence_logged(1, ["grammar_en", "grammar_de", "grammar_fr"], K=2)
for row in fixed_log:
    print(row)

pd.DataFrame(fixed_predictions).to_csv("predictions_seed1_en_de_fr.csv", index=False)
print("saved predictions CSV")

In [ ]:
orders_to_test = [
    ["grammar_en", "grammar_de", "grammar_fr"],
    ["grammar_fr", "grammar_de", "grammar_en"],
    ["grammar_de", "grammar_fr", "grammar_en"],
]

seed2_logs = []
seed2_predictions = []
for order in orders_to_test:
    log, preds = run_arrival_sequence_logged(2, order, K=2)
    seed2_logs.extend(log)
    seed2_predictions.extend(preds)
    print(f"\norder={order}")
    for row in log:
        print(" ", row)

pd.DataFrame(seed2_predictions).to_csv("predictions_seed2_all_orders.csv", index=False)
print("saved seed 2 predictions CSV")

In [ ]:
pairwise_rows = []
for seed in [0, 1, 2]:
    random.seed(seed)
    torch.manual_seed(seed)
    l_en = train_toy_lora("grammar_en", toy_tasks["grammar_en"])
    l_de = train_toy_lora("grammar_de", toy_tasks["grammar_de"])
    l_fr = train_toy_lora("grammar_fr", toy_tasks["grammar_fr"])

    pairwise_rows.append({"seed": seed, "pair": "en-de", "similarity": round(lora_similarity(l_en, l_de), 4)})
    pairwise_rows.append({"seed": seed, "pair": "en-fr", "similarity": round(lora_similarity(l_en, l_fr), 4)})
    pairwise_rows.append({"seed": seed, "pair": "de-fr", "similarity": round(lora_similarity(l_de, l_fr), 4)})
    del l_en, l_de, l_fr
    cleanup()

df_sim = pd.DataFrame(pairwise_rows)
print(df_sim)
df_sim.to_csv("pairwise_similarities.csv", index=False)

print("\naverage similarity per pair across seeds:")
print(df_sim.groupby("pair")["similarity"].mean())

In [ ]:
import random as rnd

en_subjects = ["He", "She", "It", "The dog", "The cat", "My sister", "Her brother",
               "The teacher", "The doctor", "The boy"]
en_verb_pairs = [("like", "likes"), ("want", "wants"), ("love", "loves"), ("know", "knows"),
                  ("go", "goes"), ("do", "does"), ("play", "plays"), ("eat", "eats")]
en_objects = ["the book", "the game", "the food", "the answer", "the song"]

def build_en_examples():
    rows = []
    for subj in en_subjects:
        for base, s_form in en_verb_pairs:
            for obj in en_objects:
                wrong = f"Fix the grammar: {subj} {base} {obj}."
                right = f"{subj} {s_form} {obj}."
                rows.append((wrong, right))
    return rows

de_subjects = ["Er", "Sie", "Es", "Mein Bruder", "Meine Schwester", "Der Lehrer",
               "Die Ärztin", "Der Hund", "Die Katze", "Mein Freund", "Der Fahrer",
               "Die Köchin", "Der Student", "Das Kind", "Der Nachbar"]
de_objects = ["ein Auto", "einen Hund", "eine Katze", "ein Buch", "ein Fahrrad",
              "einen Bruder", "eine Schwester", "ein Haus", "einen Ball", "eine Tasche"]

def build_de_examples():
    rows = []
    for subj in de_subjects:
        for obj in de_objects:
            wrong = f"Fix the grammar: {subj} haben {obj}."
            right = f"{subj} hat {obj}."
            rows.append((wrong, right))
    return rows

fr_subjects = ["Il", "Elle", "Mon frère", "Ma sœur", "Le professeur", "Le médecin",
               "Le chien", "Le chat", "Mon ami", "La fille"]
fr_verb_pairs = [("manger", "mange"), ("aimer", "aime"), ("parler", "parle"),
                  ("chanter", "chante"), ("jouer", "joue"), ("regarder", "regarde"),
                  ("écouter", "écoute"), ("chercher", "cherche")]
fr_objects = ["le repas", "la musique", "le film", "le jeu", "le livre"]

def build_fr_examples():
    rows = []
    for subj in fr_subjects:
        for base, s_form in fr_verb_pairs:
            for obj in fr_objects:
                wrong = f"Fix the grammar: {subj} {base} {obj}."
                right = f"{subj} {s_form} {obj}."
                rows.append((wrong, right))
    return rows

rnd.seed(42)
en_pool = list(set(build_en_examples()))
de_pool = list(set(build_de_examples()))
fr_pool = list(set(build_fr_examples()))
rnd.shuffle(en_pool); rnd.shuffle(de_pool); rnd.shuffle(fr_pool)

print("EN pool size:", len(en_pool))
print("DE pool size:", len(de_pool))
print("FR pool size:", len(fr_pool))

In [ ]:
EVAL_SIZE = 150

en_eval_large = en_pool[:EVAL_SIZE]
de_eval_large = de_pool[:EVAL_SIZE]
fr_eval_large = fr_pool[:EVAL_SIZE]

print("EN eval set:", len(en_eval_large))
print("DE eval set:", len(de_eval_large))
print("FR eval set:", len(fr_eval_large))

# sanity check: confirm zero overlap with training data
train_en_prompts = set(p for p, t in toy_tasks["grammar_en"])
train_de_prompts = set(p for p, t in toy_tasks["grammar_de"])
train_fr_prompts = set(p for p, t in toy_tasks["grammar_fr"])

overlap_en = train_en_prompts & set(p for p, t in en_eval_large)
overlap_de = train_de_prompts & set(p for p, t in de_eval_large)
overlap_fr = train_fr_prompts & set(p for p, t in fr_eval_large)

print("EN overlap with training:", len(overlap_en))
print("DE overlap with training:", len(overlap_de))
print("FR overlap with training:", len(overlap_fr))

In [ ]:
SUBSET_SIZE = 25

en_eval_subset = en_eval_large[:SUBSET_SIZE]
de_eval_subset = de_eval_large[:SUBSET_SIZE]
fr_eval_subset = fr_eval_large[:SUBSET_SIZE]

print("subset sizes:", len(en_eval_subset), len(de_eval_subset), len(fr_eval_subset))

# point eval_map (used inside run_arrival_sequence_logged) to the new subset
eval_map_subset = {
    "grammar_en": en_eval_subset,
    "grammar_de": de_eval_subset,
    "grammar_fr": fr_eval_subset,
}
eval_map_full = {
    "grammar_en": en_eval_large,
    "grammar_de": de_eval_large,
    "grammar_fr": fr_eval_large,
}
print("eval maps ready")

In [ ]:
def run_arrival_sequence_logged(seed, order_names, eval_map, K=2):
    random.seed(seed)
    torch.manual_seed(seed)

    loras = {}
    for name in order_names:
        loras[name] = train_toy_lora(name, toy_tasks[name])
    cleanup()

    collection = {}
    history = {}
    step_log = []
    all_prediction_rows = []

    for t, name in enumerate(order_names):
        incoming = loras[name]
        if len(collection) < K:
            slot = name
            collection[slot] = incoming
            history[slot] = {name}
        else:
            sims = {s: lora_similarity(collection[s], incoming) for s in collection}
            best_slot = max(sims, key=sims.get)
            n_prev = len(history[best_slot])
            collection[best_slot] = merge_loras(collection[best_slot], incoming, n_prev)
            history[best_slot].add(name)

        task_to_slot_now = {}
        for slot, tasks in history.items():
            for task in tasks:
                task_to_slot_now[task] = slot

        seen_so_far = order_names[:t+1]
        step_scores = {}
        for task in seen_so_far:
            slot = task_to_slot_now[task]
            model = load_lora_state_into_model(collection[slot])
            rows, summary = run_eval_detailed(model, eval_map[task], tag=f"seed{seed}_{'-'.join(order_names)}_step{t}_{task}")
            for r in rows:
                r["seed"] = seed
                r["order"] = "-".join(order_names)
                r["step"] = t
            all_prediction_rows.extend(rows)
            step_scores[task] = summary["avg_f1"]
            del model
            cleanup()

        step_log.append({
            "seed": seed, "step": t, "arrived": name,
            "scores_so_far": step_scores,
            "avg": round(sum(step_scores.values())/len(step_scores), 3),
            "worst": round(min(step_scores.values()), 3),
        })

    del loras, collection
    cleanup()
    return step_log, all_prediction_rows

print("updated function ready, now accepts eval_map parameter")

In [ ]:
headline_log, headline_predictions = run_arrival_sequence_logged(
    seed=0,
    order_names=["grammar_en", "grammar_de", "grammar_fr"],
    eval_map=eval_map_full,
    K=2
)

for row in headline_log:
    print(row)

pd.DataFrame(headline_predictions).to_csv("headline_run_full150_seed0.csv", index=False)
print("saved headline predictions CSV")

In [ ]:
EVAL_SIZE_MED = 50
en_eval_med = en_pool[:EVAL_SIZE_MED]
de_eval_med = de_pool[:EVAL_SIZE_MED]
fr_eval_med = fr_pool[:EVAL_SIZE_MED]

eval_map_med = {
    "grammar_en": en_eval_med,
    "grammar_de": de_eval_med,
    "grammar_fr": fr_eval_med,
}

orders_to_test = [
    ["grammar_en", "grammar_de", "grammar_fr"],
    ["grammar_fr", "grammar_de", "grammar_en"],
    ["grammar_de", "grammar_fr", "grammar_en"],
]

full_sweep_logs = []
full_sweep_predictions = []

for seed in [0, 1, 2]:
    for order in orders_to_test:
        log, preds = run_arrival_sequence_logged(seed, order, eval_map_med, K=2)
        full_sweep_logs.extend(log)
        full_sweep_predictions.extend(preds)
        print(f"seed={seed}, order={order}")
        for row in log:
            print(" ", row)

pd.DataFrame(full_sweep_predictions).to_csv("full_sweep_50ex_predictions.csv", index=False)
pd.DataFrame(full_sweep_logs).to_csv("full_sweep_50ex_logs.csv", index=False)
print("saved full sweep CSVs")

In [ ]:
import difflib
import math
from collections import Counter

def get_ngrams(tokens, n):
    return Counter([tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)])

def gleu_score(pred, target):
    """
    Sentence-level GLEU (Google's BLEU variant used for grammar correction).
    Measures overlap of n-grams between prediction and reference.
    """
    pred_tokens = clean(pred)
    target_tokens = clean(target)
    if not pred_tokens or not target_tokens:
        return 0.0
    scores = []
    for n in [1, 2]:
        pred_ng = get_ngrams(pred_tokens, n)
        target_ng = get_ngrams(target_tokens, n)
        if not pred_ng or not target_ng:
            scores.append(0.0)
            continue
        overlap = sum((pred_ng & target_ng).values())
        precision = overlap / max(sum(pred_ng.values()), 1)
        recall = overlap / max(sum(target_ng.values()), 1)
        if precision + recall == 0:
            scores.append(0.0)
        else:
            scores.append(2 * precision * recall / (precision + recall))
    return round(sum(scores) / len(scores), 3)

def run_eval_detailed(model, eval_set, tag=""):
    rows = []
    for prompt, target in eval_set:
        out = generate(model, prompt)
        rows.append({
            "task": tag,
            "input": prompt,
            "expected": target,
            "prediction": out,
            "exact_match": 1.0 if clean(out) == clean(target) else 0.0,
            "edit_acc": round(difflib.SequenceMatcher(
                None, out.strip().lower(), target.strip().lower()).ratio(), 3),
            "f1": round(score(out, target), 3),
            "gleu": gleu_score(out, target),
        })
    n = len(rows)
    summary = {
        "avg_exact_match": round(sum(r["exact_match"] for r in rows)/n, 3),
        "avg_edit_acc":    round(sum(r["edit_acc"]    for r in rows)/n, 3),
        "avg_f1":          round(sum(r["f1"]          for r in rows)/n, 3),
        "avg_gleu":        round(sum(r["gleu"]        for r in rows)/n, 3),
    }
    return rows, summary

print("All 4 metrics ready: exact_match, edit_acc, f1, gleu")

In [ ]:
EVAL_SIZE_100 = 100
en_eval_100 = en_pool[:EVAL_SIZE_100]
de_eval_100 = de_pool[:EVAL_SIZE_100]
fr_eval_100 = fr_pool[:EVAL_SIZE_100]

eval_map_100 = {
    "grammar_en": en_eval_100,
    "grammar_de": de_eval_100,
    "grammar_fr": fr_eval_100,
}

orders_to_test = [
    ["grammar_en", "grammar_de", "grammar_fr"],
    ["grammar_fr", "grammar_de", "grammar_en"],
    ["grammar_de", "grammar_fr", "grammar_en"],
]

sweep100_logs = []
sweep100_predictions = []

for seed in [0, 1, 2]:
    for order in orders_to_test:
        log, preds = run_arrival_sequence_logged(
            seed, order, eval_map_100, K=2
        )
        for row in log:
            row["order"] = "-".join(order)
        sweep100_logs.extend(log)
        sweep100_predictions.extend(preds)
        print(f"\nseed={seed}, order={'-'.join(order)}")
        for row in log:
            print(" ", row)

pd.DataFrame(sweep100_predictions).to_csv(
    "sweep_100ex_all_metrics_predictions.csv", index=False)
pd.DataFrame(sweep100_logs).to_csv(
    "sweep_100ex_all_metrics_logs.csv", index=False)
print("\nsaved all CSVs")